# Part 3 – Agentic RAG (Iterative Loops, Reflection, and Tool Orchestration)

While Self-RAG embeds reflection tokens directly into model generation and CRAG evaluates retrieval quality once right after a database fetch, Agentic RAG takes a structural leap forward: it treats the entire RAG pipeline as an autonomous agentic loop.

Instead of treating retrieval as a single, static step (Query $\rightarrow$ Search $\rightarrow$ Answer), Agentic RAG models retrieval as an iterative, multi-pass tool orchestration problem with query planning, tool selection, loop execution, and self-correction.

## 1. The Core Shift: From Pipeline to Agent Loop
In a standard or even advanced RAG setup, the system is reactive. In Agentic RAG, the system is proactive and goal-driven. It can:

**Deconstruct complex queries:** Break a multi-part user question into a sequence of targeted sub-queries.

**Select optimal tools dynamically:** Choose whether to use a dense vector search, a sparse keyword search (BM25), a structured Knowledge Graph traversal (Neo4j/Cypher), or an external web search API based on the needs of the sub-query.

**Inspect and reflect:** Review retrieved results during execution. If a search result is insufficient, the agent reformulates its query and searches again (multi-pass refinement) before generating the final answer.


## 2. The Core Components of an Agentic RAG Architecture


In [ ]:
[User Complex Query] 
        │
        v
[1. Planner / ReAct Agent] <────────────────────────┐
        │                                            │
        ├─► [Tool A: Vector DB Search]               │
        ├─► [Tool B: Knowledge Graph (Cypher)]       │
        └─► [Tool C: Web Fallback Search]            │
        │                                            │
        v                                            │
[2. Evaluator / Reflection Agent] ──(Incomplete?)───►┘
        │
    (Complete?)
        │
        v
[3. Final Synthesizer / Generator]

### A. The Planner / ReAct Agent
When a user asks a difficult question (e.g., "Compare TechCorp's European supply chain bottlenecks in Q3 with its competitor DataStream's logistics routes"), the agent builds an execution plan. It breaks the task down into sequential steps rather than firing a single blunt query.

### B. Tool Orchestration
The agent has access to multiple specialized retrieval tools and decides which one to invoke:

**Vector Search Tool:** Best for semantic exploration and conceptual paragraphs.

**Graph Traversal Tool:** Best for multi-hop structural connections ("Who owns X, which partners with Y?").

**SQL / Metadata Filter Tool:** Best for exact date filtering or numerical aggregates.

### C. Reflection & Multi-Pass Loop
After executing a tool call, the agent pauses to inspect the output:

Did this return enough concrete facts?

**Are there missing data points?**
If the information is incomplete, the agent dynamically generates a new, refined query and queries a secondary tool. This loop repeats until the agent has sufficient context or hits a max-iteration guardrail.

## 3. Implementation Pattern: Agentic RAG Workflow (agentic_rag_loop.py)
Here is a conceptual implementation of an iterative Agentic RAG loop using a structured Python control flow:

In [ ]:
"""
agentic_rag_loop.py
Demonstrates an iterative Agentic RAG loop with query planning, 
tool orchestration, and reflection-based multi-pass refinement.
"""

from typing import List, Dict, Any

class AgenticRAGOrchestrator:
    def __init__(self):
        print("Initializing Agentic RAG Orchestrator with Multi-Tool Registry...")
        self.max_iterations = 3

    def tool_vector_search(self, query: str) -> str:
        """Simulates searching a vector database."""
        print(f"   [Tool Execution] Vector Search for: '{query}'")
        if "supply chain" in query.lower():
            return "TechCorp Europe experienced supply chain bottlenecks in Q3 due to port strikes in Berlin."
        return "No direct semantic matches found in vector index."

    def tool_graph_traversal(self, entity: str) -> str:
        """Simulates traversing a Neo4j knowledge graph."""
        print(f"   [Tool Execution] Graph Traversal for entity: '{entity}'")
        if "techcorp" in entity.lower():
            return "Graph Path: (TechCorp Global) -[OWNS]-> (TechCorp Europe) -[PARTNERS_WITH]-> (DataStream Logistics)"
        return "Entity not found in Knowledge Graph."

    def evaluate_sufficiency(self, context_accumulated: List[str], goal: str) -> bool:
        """Reflection agent checks if gathered context is sufficient to answer the goal."""
        print(f"   [Reflection Agent] Evaluating sufficiency of {len(context_accumulated)} collected facts...")
        # If we have both supply chain details and partner details, we are satisfied
        combined_text = " ".join(context_accumulated)
        if "port strikes" in combined_text and "DataStream Logistics" in combined_text:
            return True
        return False

    def run_agentic_loop(self, user_query: str) -> str:
        """Executes the autonomous plan-execute-reflect loop."""
        print(f"\n--- Starting Agentic RAG Loop for: '{user_query}' ---")
        
        context_bucket = []
        iteration = 0
        
        # Step 1: Initial Plan Formulation
        current_sub_query = "TechCorp supply chain Q3"
        
        while iteration < self.max_iterations:
            iteration += 1
            print(f"\n[Iteration {iteration}/{self.max_iterations}]")
            
            # Step 2: Tool Selection & Execution
            if iteration == 1:
                result = self.tool_vector_search(current_sub_query)
                context_bucket.append(result)
            elif iteration == 2:
                # Based on reflection, agent decides it needs structural relationship data
                result = self.tool_graph_traversal("TechCorp Europe")
                context_bucket.append(result)
                
            # Step 3: Reflection Check
            if self.evaluate_sufficiency(context_bucket, user_query):
                print("   [Reflection Agent] Context is sufficient! Proceeding to generation.")
                break
            else:
                print("   [Reflection Agent] Context insufficient. Refining plan for next pass...")
                current_sub_query = "TechCorp logistics partners Germany"

        # Step 4: Final Synthesis Mock
        final_answer = f"Generated Answer based on agentic multi-pass context: {' '.join(context_bucket)}"
        return final_answer

# --- Execution Block ---
if __name__ == "__main__":
    agent = AgenticRAGOrchestrator()
    query = "What caused TechCorp's Q3 supply chain issues and who are their key logistics partners?"
    
    response = agent.run_agentic_loop(query)
    print("\n--- Final Agent Response ---")
    print(response)

## 4. Why Agentic RAG is the State-of-the-Art in Enterprise Systems
**Handles Ambiguous & Multi-Part User Prompts:** Users rarely ask clean, single-vector questions. Agents excel at parsing complex, multi-intent queries into structured step-by-step tool execution plans.

**Resilience Through Multi-Pass Iteration:** If the first search returns weak or partial results, the agent doesn't fail or hallucinate—it pivots, rewrites its query, and tries a different tool.

**Maximized Cost-to-Accuracy Ratio:** While agents use more tokens per query due to iterative planning loops, they drastically reduce enterprise failure rates on complex reasoning tasks where standard RAG completely breaks down.